# AccessAI demo
A short demo of AccessAI: a simple accessibility scanner and conservative fixer. The goal is to show how the pipeline works with local HTML pages.


# Setup
This notebook runs locally and on Kaggle without external API keys. It demonstrates the scanner, fixer, and patcher using the sample pages provided in the repo. Follow cells in order and run them top-to-bottom.


In [8]:
# Try importing BeautifulSoup, install requirements if missing (best-effort)
import sys, os
try:
    import bs4
    print('bs4 available')
except Exception:
    print('bs4 not found, installing requirements (best-effort)')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'capstone-project/requirements.txt'])
    import bs4
    print('installed requirements')


bs4 available


In [9]:
# Import project agents
from agents import scanner, fixer, patcher

# Lightweight code_exec helper (demo-only)
def code_exec(code: str):
    """Run a short Python snippet and return stdout+returncode.
    This is a demo helper used only for small verification snippets.
    """
    import subprocess, sys
    try:
        res = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=5)
        return {'stdout': res.stdout, 'stderr': res.stderr, 'returncode': res.returncode}
    except Exception as e:
        return {'stdout': '', 'stderr': str(e), 'returncode': -1}

print('agents imported:', scanner.__name__, fixer.__name__, patcher.__name__)


agents imported: agents.scanner agents.fixer agents.patcher


## Demo helper: run scan, suggest fixes, apply patches
This helper reads an HTML file, runs the scanner, asks the fixer for conservative suggestions (with optional verification), and prints short, readable results. The goal is to make the demo easy to run in other Notebook editors and compilers.


In [10]:
from pathlib import Path
import html

def demo_file(path):
    path = Path(path)
    html_text = path.read_text(encoding='utf-8')
    print(f'--- Demo: {path.name} ---')
    base_scan = scanner.analyze_html(html_text, str(path))
    print('Baseline:', base_scan.get('summary'))
    for issue in base_scan.get('issues', [])[:10]:
        print('-', issue['id'], '-', issue['message'])

    suggestions = fixer.suggest_fixes(base_scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
    if not suggestions:
        print('No automatic suggestions produced.')
        return base_scan

    for i, s in enumerate(suggestions, start=1):
        issue = s.get('issue', {})
        print(f'\nSuggestion {i}: {s.get("suggestion")}')
        print(' Issue:', issue.get('id'), '-', issue.get('message'))
        print(' Patch (human-readable):', s.get('patch'))
        if 'verification' in s:
            print(' Verification stdout:', s['verification'].get('stdout'))
        if 'post_scan' in s:
            post = s['post_scan']
            print(' Post-scan summary:', post.get('summary'))
            print(' Reduction:', s.get('reduction'))
            # show a small snippet of the patched HTML for context
            patched = patcher.apply_patch_to_html(html_text, issue, s.get('patch', ''))
            snippet = patched[:500].replace('\n',' ')
            print(' Patched snippet:', snippet[:400] + ('...' if len(snippet)>400 else ''))

    return suggestions


## Run the demo on included sample pages
The repository includes several sample pages in `capstone-project/data/sample_pages/`. The cells below run the demo helper on each page and print concise results.


In [12]:
# Resolve repository root and locate sample pages directory
from pathlib import Path
p = Path.cwd()
repo_root = p
# Walk upward to find the folder containing `capstone-project`
for _ in range(10):
    if (repo_root / 'capstone-project').exists():
        break
    if repo_root.parent == repo_root:
        break
    repo_root = repo_root.parent

pages_dir = repo_root / 'capstone-project' / 'data' / 'sample_pages'
print('pages_dir ->', pages_dir)

pages = sorted([x for x in pages_dir.glob('*.html')])
print('Found', len(pages), 'sample pages')


pages_dir -> /workspaces/fraudshield-workforce/capstone-project/data/sample_pages
Found 7 sample pages


## Interactive patch preview
Use the widget below to review suggested fixes, toggle which suggestions to apply, and preview the patched HTML before saving.


In [14]:
# Interactive preview helper (disabled for reproducible, non-interactive execution)
preview = False

def show_preview(html_text, title=None):
    # No-op preview during automated runs; local runs can set `preview=True`.
    if preview:
        try:
            from IPython.display import display, HTML
            display(HTML(html_text))
        except Exception:
            pass

print('Preview helper loaded; preview=', preview)


Preview helper loaded; preview= False


In [15]:
from pathlib import Path

# Create patched version for a sample and save to file, then print a short preview
p = Path('capstone-project/data/sample_pages/bad.html')
if not p.exists():
    p = find_repo_root() / 'capstone-project' / 'data' / 'sample_pages' / 'bad.html'
html_text = p.read_text(encoding='utf-8')
scan = scanner.analyze_html(html_text, str(p))
suggestions = fixer.suggest_fixes(scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
patched = html_text
for s in suggestions:
    patched = patcher.apply_patch_to_html(patched, s.get('issue', {}), s.get('patch', ''))

out_path = Path('capstone-project/tmp/patched_bad.html')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(patched, encoding='utf-8')
print('Patched HTML written to', out_path)
preview = patched[:4000]
if len(patched) > 4000:
    preview += '\n\n... (truncated) ...\n'
print(preview)


Patched HTML written to capstone-project/tmp/patched_bad.html
<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Bad sample</title>
</head>
<body><h1>Page title</h1>
<h2>Subtitle without H1</h2>
<p>Images without alt and unlabeled inputs.</p>
<img alt="Describe image" src="/img/missing.png"/>
<form>
<label>Label</label><input aria-label="Label" type="text"/>
</form>
</body>
</html>



In [16]:
# FOr medium file.html
from pathlib import Path
sample = 'medium.html'
repo = globals().get('repo_root', Path.cwd())
pp = repo / 'capstone-project' / 'data' / 'sample_pages' / sample
if not pp.exists():
    pp = Path('capstone-project/data/sample_pages') / sample
    if not pp.exists():
        raise FileNotFoundError(pp)

html_text = pp.read_text(encoding='utf-8')
scan = scanner.analyze_html(html_text, str(pp))
print('Scan summary:', scan.get('summary'))

suggestions = fixer.suggest_fixes(scan, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
if not suggestions:
    print('No suggestions produced for', sample)
else:
    print(f"\nFound {len(suggestions)} suggestion(s):")
    for i, s in enumerate(suggestions, start=1):
        issue = s.get('issue', {})
        print(f"\nSuggestion {i} — Issue: {issue.get('id')} | Message: {issue.get('message')}")
        print(' Suggestion text:', s.get('suggestion'))
        print(' Patch snippet:', (s.get('patch') or '')[:300].replace('\n',' '))
        if 'post_scan' in s:
            print(' Post-scan summary:', s['post_scan'].get('summary'))
        if 'verification' in s:
            print(' Verification stdout:', s['verification'].get('stdout'))

# Apply all suggestions conservatively and save patched file
patched = html_text
for s in suggestions:
    patched = patcher.apply_patch_to_html(patched, s.get('issue', {}), s.get('patch', ''))

out_path = Path('capstone-project/tmp/patched_medium.html')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(patched, encoding='utf-8')
print('\nPatched HTML written to', out_path)

preview = patched[:3000]
if len(patched) > 3000:
    preview += '\n\n... (truncated) ...\n'
print('\n--- Patched preview (first 3000 chars) ---')
print(preview)


Scan summary: 3 issue(s) found

Found 3 suggestion(s):

Suggestion 1 — Issue: alt_missing | Message: Image missing alt text
 Suggestion text: Add alt attribute: alt="Describe image"
 Patch snippet: add alt to img:nth-of-type(2)
 Post-scan summary: 2 issue(s) found
 Verification stdout: fail

Suggestion 2 — Issue: heading_structure | Message: Document should start with an H1
 Suggestion text: Ensure the page has a single H1 at top
 Patch snippet: <h1>Page title</h1>
 Post-scan summary: 2 issue(s) found
 Verification stdout: ok

Suggestion 3 — Issue: label_missing | Message: Form control with id "email" missing label or aria-label
 Suggestion text: Add associated <label> or aria-label to form control
 Patch snippet: insert label for control
 Post-scan summary: 2 issue(s) found

Patched HTML written to capstone-project/tmp/patched_medium.html

--- Patched preview (first 3000 chars) ---
<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Medium sample</title>
</head>
<body><h1>Pa

In [17]:
# Run scoring across all sample pages (includes newly added pages)
from agents.eval import score_page, apply_patches_and_rescan
from pathlib import Path
results = []
pages_dir = repo_root / 'capstone-project' / 'data' / 'sample_pages'
for p in sorted(pages_dir.iterdir()):
    try:
        html_text = p.read_text(encoding='utf-8')
    except Exception:
        html_text = ''
    base = scanner.analyze_html(html_text, str(p))
    fixes = fixer.suggest_fixes(base, html_text, tools={'scanner': scanner.analyze_html, 'code_exec': code_exec})
    final_html, post = apply_patches_and_rescan(html_text, fixes, scanner.analyze_html)
    sc = score_page(base, post)
    results.append({'file': p.name, 'before': sc['total_before'], 'after': sc['total_after'], 'reduction': sc['total_reduction']})
    out_path = Path('capstone-project/tmp') / f'patched_{p.name}'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(final_html, encoding='utf-8')

print('Summary:')
for r in results:
    print(f"{r['file']}: before={r['before']} after={r['after']} reduction={r['reduction']}")

Summary:
bad.html: before=3 after=0 reduction=3
bad_contrast.html: before=0 after=0 reduction=0
form_unlabeled.html: before=3 after=0 reduction=3
good.html: before=0 after=0 reduction=0
malformed_snippet.html: before=1 after=0 reduction=1
medium.html: before=3 after=0 reduction=3
missing_alt_complex.html: before=3 after=0 reduction=3
